# Plankton dataset class overlap

For the 7 plankton datasets used in Setup (iii) of the paper (1 in-domain training set + 6 cross-domain test sets), how many classes do they share?

Two views:
- **Species-level**: normalised exact-string match of class names.
- **Genus-level**: collapse to the first taxonomic token (e.g. *Chaetoceros_socialis*, *Chaetoceros_curvisetus* both become *chaetoceros*).

Matters because plankton vocabularies disagree on suffixes (`_spp`, `_sp`, `_single`, `_chain`, ...), instrument-specific morphotypes, and prefixes (PlanktoShare prefixes everything with `Phyto_` or similar), so literal string match under-reports biological overlap.

In [ ]:
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

DATASETS = {
    'WCO-L4-IFCB':   'danielaivanova/wco-l4-ifcb-pml',
    'SYKE-IFCB':     'danielaivanova/syke-plankton-ifcb-2022',
    'SYKE-ZooScan':  'danielaivanova/syke-plankton-zooscan-2024',
    'PlanktoShare':  'danielaivanova/plankto-share',
    'DAP-IFCB':      'danielaivanova/daplankton-lab-ifcb',
    'DAP-CS':        'danielaivanova/daplankton-lab-cs',
    'DAP-FC':        'danielaivanova/daplankton-lab-fc',
}

## Load class lists

Pull only the `label` column metadata via streaming, no image bytes touched.

In [ ]:
raw_classes = {}
for short, repo in DATASETS.items():
    ds = load_dataset(repo, split='train', streaming=True)
    raw_classes[short] = list(ds.features['label'].names)
    print(f'{short:15s} {len(raw_classes[short]):4d} classes  (first 3: {raw_classes[short][:3]})')

## Normalisation

- **species-level**: lowercase, replace `_`/`-`/whitespace with a single space, strip PlanktoShare's `phyto_`/`zoo_` prefixes, strip trailing morphotype/colony/etc. suffixes that vary across datasets.
- **genus-level**: take the first whitespace-separated token of the species-normalised name.

In [ ]:
_SUFFIX_RE = re.compile(
    r'\s+(?:spp|sp|morphotype\d*|single|double|chain|complete|multiple|fragment|intact|colony|like|drop|small|cell|cells|coiled|type)$'
)
_PREFIX_RE = re.compile(r'^(?:phyto|zoo)\s+')

def normalise_species(name):
    s = name.lower().replace('_', ' ').replace('-', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = _PREFIX_RE.sub('', s)
    prev = None
    while prev != s:
        prev = s
        s = _SUFFIX_RE.sub('', s).strip()
    return s

def normalise_genus(name):
    s = normalise_species(name)
    if not s:
        return s
    return s.split(' ', 1)[0]

samples = ['Chaetoceros_socialis', 'Phyto_Chaetoceros', 'Cryptophyceae_drop',
           'Copepoda_calanoida', 'Bivalvia_multiple', 'Cerautulina_pelagica_single_double',
           'unknown_spherical', 'Dictyocha_speculum']
for s in samples:
    print(f'  {s:40s} -> species={normalise_species(s):35s} genus={normalise_genus(s)}')

## Build normalised class sets

In [ ]:
species_sets = {short: set(filter(None, (normalise_species(c) for c in cls)))
                for short, cls in raw_classes.items()}
genus_sets   = {short: set(filter(None, (normalise_genus(c)   for c in cls)))
                for short, cls in raw_classes.items()}

summary = pd.DataFrame({
    'raw classes':     {k: len(v) for k, v in raw_classes.items()},
    'unique species':  {k: len(v) for k, v in species_sets.items()},
    'unique genera':   {k: len(v) for k, v in genus_sets.items()},
})
summary

## Pairwise overlap matrices

For each pair (A, B):
- |A intersect B| - raw shared-class count (diagonal = |A|)
- Jaccard = |A intersect B| / |A union B| - symmetric similarity in [0, 1]

In [ ]:
def overlap_matrices(sets):
    names = list(sets.keys())
    n = len(names)
    inter = np.zeros((n, n), dtype=int)
    jacc  = np.zeros((n, n))
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            A, B = sets[a], sets[b]
            inter[i, j] = len(A & B)
            union = len(A | B)
            jacc[i, j]  = (len(A & B) / union) if union else 0.0
    return names, inter, jacc

names, sp_inter, sp_jacc = overlap_matrices(species_sets)
_,     gn_inter, gn_jacc = overlap_matrices(genus_sets)

In [ ]:
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

print('SPECIES-LEVEL pairwise intersection (rows=A, cols=B; diagonal = |A|):')
display(pd.DataFrame(sp_inter, index=names, columns=names))

print('\nSPECIES-LEVEL Jaccard:')
display(pd.DataFrame(sp_jacc, index=names, columns=names))

print('\nGENUS-LEVEL pairwise intersection:')
display(pd.DataFrame(gn_inter, index=names, columns=names))

print('\nGENUS-LEVEL Jaccard:')
display(pd.DataFrame(gn_jacc, index=names, columns=names))

## Heatmaps

In [ ]:
def plot_heatmap(ax, mat, names, title, fmt='{:d}', cmap='viridis', vmin=None, vmax=None):
    im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(names)))
    ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right')
    ax.set_yticklabels(names)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            txt = fmt.format(val)
            colour = 'white' if (vmax is not None and val > vmax * 0.55) else 'black'
            ax.text(j, i, txt, ha='center', va='center', color=colour, fontsize=8)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
plot_heatmap(axes[0, 0], sp_inter, names,
             'Species-level: shared class count', fmt='{:d}', cmap='Blues',
             vmin=0, vmax=int(sp_inter.max()))
plot_heatmap(axes[0, 1], sp_jacc, names,
             'Species-level: Jaccard similarity', fmt='{:.2f}', cmap='Blues',
             vmin=0, vmax=1)
plot_heatmap(axes[1, 0], gn_inter, names,
             'Genus-level: shared genus count', fmt='{:d}', cmap='Purples',
             vmin=0, vmax=int(gn_inter.max()))
plot_heatmap(axes[1, 1], gn_jacc, names,
             'Genus-level: Jaccard similarity', fmt='{:.2f}', cmap='Purples',
             vmin=0, vmax=1)
plt.suptitle('Plankton dataset class overlap (WCO-L4-IFCB is the in-domain training set)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Per-dataset coverage bars

For each dataset A, how many of its classes (species and genera) are also present in each of the other 6 datasets? Expressed as a percentage of |A|.

In [ ]:
def coverage_bars(sets, names, kind):
    n = len(names)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4), sharey=True)
    for ax, A in zip(axes, names):
        others = [B for B in names if B != A]
        share_pct = [100.0 * len(sets[A] & sets[B]) / max(1, len(sets[A])) for B in others]
        bars = ax.bar(others, share_pct,
                      color='steelblue' if kind == 'species' else 'rebeccapurple')
        ax.set_title(f'{A}\n(|A|={len(sets[A])} {kind})', fontsize=10)
        ax.set_xticklabels(others, rotation=45, ha='right', fontsize=8)
        ax.set_ylim(0, 100)
        ax.grid(axis='y', alpha=0.3)
        for b, v in zip(bars, share_pct):
            ax.text(b.get_x() + b.get_width() / 2, v + 1, f'{v:.0f}%',
                    ha='center', fontsize=7)
    axes[0].set_ylabel(f'% of A\'s {kind} also in B')
    plt.suptitle(f'{kind.title()}-level: how much of dataset A is covered by each other dataset',
                 fontsize=12, y=1.04)
    plt.tight_layout()
    plt.show()

coverage_bars(species_sets, names, 'species')
coverage_bars(genus_sets,   names, 'genus')

## Concrete shared / unique classes

In [ ]:
print('=== Species shared between WCO-L4-IFCB and each other dataset ===')
A = species_sets['WCO-L4-IFCB']
for B_name in names:
    if B_name == 'WCO-L4-IFCB':
        continue
    shared = sorted(A & species_sets[B_name])
    print(f'\n-- WCO ∩ {B_name}: {len(shared)} species --')
    for s in shared[:30]:
        print(f'   {s}')
    if len(shared) > 30:
        print(f'   ... and {len(shared) - 30} more')

In [ ]:
print('=== Genera shared between WCO-L4-IFCB and each other dataset ===')
A = genus_sets['WCO-L4-IFCB']
for B_name in names:
    if B_name == 'WCO-L4-IFCB':
        continue
    shared = sorted(A & genus_sets[B_name])
    print(f'\n-- WCO ∩ {B_name}: {len(shared)} genera --')
    print('  ' + ', '.join(shared))

In [ ]:
wco_only_species = species_sets['WCO-L4-IFCB'] - set().union(
    *(species_sets[b] for b in names if b != 'WCO-L4-IFCB'))
wco_only_genus = genus_sets['WCO-L4-IFCB'] - set().union(
    *(genus_sets[b] for b in names if b != 'WCO-L4-IFCB'))
print(f'Species unique to WCO: {len(wco_only_species)} / {len(species_sets["WCO-L4-IFCB"])}')
print(f'Genera  unique to WCO: {len(wco_only_genus)} / {len(genus_sets["WCO-L4-IFCB"])}')
print('\nFirst 30 WCO-only genera:')
for g in sorted(wco_only_genus)[:30]:
    print(f'  {g}')

In [ ]:
import io
import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

RESULTS_DIR = Path('fl_results')
ZIPS = {
    'CIFAR-10':       RESULTS_DIR / 'table3_jsons.zip',
    'WCO-L4-IFCB':    RESULTS_DIR / 'table4_jsons.zip',
}
# The OOD setup (CIFAR-trained, evaluated on plankton/satellite/etc.) lives in
# the same JSONs as the CIFAR-trained ones — it's identified by the eval_dataset
# field, not the train dataset. So we don't need a separate zip for it.

# Per-backbone labels (match the aggregator).
BACKBONE_LABELS = {
    'resnet18':  'ResNet-18',
    'clip_b16':  'CLIP-B/16',
    'dino_s16':  'DINOv3-S/16',
    'dino_b16':  'DINOv3-B/16',
    'tips_b14':  'TIPS-B/14',
}

# Filename pattern: capture (backbone_tag, train_dataset, cell, seed).
FNAME_RE = re.compile(
    r'fed_unsup_simclr_(?P<bb>[a-z0-9_]+?)_'
    r'(?P<train>cifar10|cifar100|wco_l4_ifcb_pml|syke_ifcb)_'
    r'(?:(?P=bb)_)?'                # optional duplicated tag from --run_name
    r'(?P<cell>\d+C_\d+pct)_seed(?P<seed>\d+)_\d{8}_\d{6}\.json$'
)

# Map each eval_dataset to the "experimental setup" it belongs to.
SETUP_OF_EVAL = {
    # Setup (i): in-domain natural images. Train on CIFAR-10.
    'cifar10':              'In-domain (natural images)',
    'cifar100':             'In-domain (natural images)',
    'tiny':                 'In-domain (natural images)',
    # Setup (ii): cross-modality OOD. Train on CIFAR-10.
    'whoi_plankton':        'Cross-modality OOD',
    'eurosat':              'Cross-modality OOD',
    'bone_marrow':          'Cross-modality OOD',
    'wikiart_style':        'Cross-modality OOD',
    # Setup (iii): plankton-specific. Train on WCO-L4-IFCB.
    'wco_l4_ifcb_pml':      'Plankton (domain-specific)',
    # 'syke_ifcb':            'Plankton (domain-specific)',
    'syke_zooscan':         'Plankton (domain-specific)',
    'plankto_share':        'Plankton (domain-specific)',
    'daplankton_lab_ifcb':  'Plankton (domain-specific)',
    'daplankton_lab_cs':    'Plankton (domain-specific)',
    'daplankton_lab_fc':    'Plankton (domain-specific)',
}

METRIC_KEYS = ['linear_accuracy', 'knn_accuracy', 'cluster_accuracy',
               'prototype_accuracy', 'silhouette_score']

def parse_json_blob(blob: bytes, train_dataset: str, fname: str):
    """Yield long-format rows (dict) for a single JSON file."""
    d = json.loads(blob.decode('utf-8'))
    m = FNAME_RE.search(fname)
    if not m:
        return
    bb = m['bb']
    cell = m['cell']
    seed = int(m['seed'])
    # Linear / eval_only accuracy lives in eval_results;
    # k-NN / cluster / prototype / silhouette in unsup_results.
    by_eval = {}
    for entry in d.get('eval_results', []):
        ev = entry.get('eval_dataset', train_dataset)
        by_eval.setdefault(ev, {}).update({
            k: entry.get(k) for k in ('linear_accuracy', 'eval_only_accuracy')
        })
    for entry in d.get('unsup_results', []):
        ev = entry.get('eval_dataset', train_dataset)
        by_eval.setdefault(ev, {}).update({
            k: entry.get(k) for k in ('knn_accuracy', 'cluster_accuracy',
                                       'prototype_accuracy', 'silhouette_score')
        })
    for ev, mdict in by_eval.items():
        setup = SETUP_OF_EVAL.get(ev, f'unknown ({ev})')
        for k in METRIC_KEYS:
            if mdict.get(k) is None:
                continue
            yield {
                'setup':         setup,
                'train_dataset': train_dataset,
                'eval_dataset':  ev,
                'backbone':      BACKBONE_LABELS.get(bb, bb),
                'backbone_tag':  bb,
                'cell':          cell,
                'seed':          seed,
                'metric':        k,
                'value':         float(mdict[k]),
            }

rows = []
for train_dataset, zip_path in ZIPS.items():
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            if not info.filename.endswith('.json'):
                continue
            rows.extend(parse_json_blob(zf.read(info), train_dataset, info.filename))

df = pd.DataFrame(rows)

# Sanity check: every (setup, backbone, cell, eval_dataset, metric) should
# have 1-3 seed values; we'll aggregate to mean+std for plotting.
print(f'Total rows: {len(df)}')
print(f'Setups: {sorted(df.setup.unique())}')
print(f'Backbones: {sorted(df.backbone.unique())}')
print(f'Cells: {sorted(df.cell.unique())}')
print(f'Metrics: {sorted(df.metric.unique())}')

# Aggregate over seeds.
agg = (df.groupby(['setup', 'backbone', 'backbone_tag', 'cell', 'eval_dataset', 'metric'])
         .value.agg(['mean', 'std', 'count'])
         .reset_index())
agg.head()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams


# ----- styling ------------------------------------------------------------
rcParams.update({
    'figure.facecolor':       'white',
    'axes.facecolor':         'white',
    'axes.edgecolor':         '#cccccc',
    'axes.linewidth':         0.8,
    'axes.grid':              True,
    'grid.color':             '#dddddd',
    'grid.linewidth':         0.6,
    'axes.axisbelow':         True,
    'xtick.color':            '#333333',
    'ytick.color':            '#333333',
    'axes.labelcolor':        '#333333',
    'font.family':            'serif',
    'font.serif':             ['Liberation Serif', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset':       'stix',
    'savefig.facecolor':      'white',
    'savefig.bbox':           'tight',
    'savefig.pad_inches':     0.05,
})


# ----- constants ----------------------------------------------------------
METRIC_ORDER = ['linear_accuracy', 'knn_accuracy', 'cluster_accuracy',
                'prototype_accuracy', 'silhouette_score']
METRIC_LABELS = {
    'linear_accuracy':    'Linear',
    'knn_accuracy':       'k-NN',
    'cluster_accuracy':   'K-Means',
    'prototype_accuracy': 'Prototype',
    'silhouette_score':   'Silhouette',
}
BACKBONES_ORDER = ['ResNet-18', 'CLIP-B/16', 'DINOv3-S/16', 'DINOv3-B/16', 'TIPS-B/14']
BACKBONE_COLOURS = {
    'ResNet-18':    '#9e9e9e',
    'CLIP-B/16':    '#5c8ec7',
    'DINOv3-S/16':  '#5bba6f',
    'DINOv3-B/16':  '#2e7d44',
    'TIPS-B/14':    '#d96b6b',
}

SETUPS_ORDER = [
    'In-domain (natural images)',
    'Cross-modality OOD',
    'Plankton (domain-specific)',
]
CELLS_ORDER = ['200C_1pct', '200C_5pct', '2000C_1pct', '2000C_5pct']
CELL_TITLES = {
    '200C_1pct':   '200C / 1%',
    '200C_5pct':   '200C / 5%',
    '2000C_1pct':  '2000C / 1%',
    '2000C_5pct':  '2000C / 5%',
}
EVAL_TITLES = {
    'cifar10': 'CIFAR-10',
    'cifar100': 'CIFAR-100',
    'tiny': 'Tiny-IN',
    'whoi_plankton': 'WHOI',
    'eurosat': 'EuroSAT',
    'bone_marrow': 'Bone Marrow',
    'wikiart_style': 'WikiArt',
    'wco_l4_ifcb_pml': 'WCO-L4-IFCB',
    # 'syke_ifcb': 'SYKE-IFCB',
    'syke_zooscan': 'SYKE-ZooScan',
    'plankto_share': 'PlanktoShare',
    'daplankton_lab_ifcb': 'DAP-IFCB',
    'daplankton_lab_cs': 'DAP-CytoSense',
    'daplankton_lab_fc': 'DAP-FlowCam',
}

SILHOUETTE_SCALE = lambda v: np.clip((v + 0.1) / 0.5, 0, 1)


# ----- the panel ----------------------------------------------------------
def radial_bar_panel(ax, setup, eval_dataset, cell, title=None):
    sub = agg[(agg.setup == setup)
              & (agg.eval_dataset == eval_dataset)
              & (agg.cell == cell)]
    if sub.empty:
        ax.set_visible(False)
        return

    pivot_mean = (sub.pivot_table(index='backbone', columns='metric',
                                  values='mean', aggfunc='first')
                     .reindex(index=BACKBONES_ORDER, columns=METRIC_ORDER))
    pivot_std  = (sub.pivot_table(index='backbone', columns='metric',
                                  values='std',  aggfunc='first')
                     .reindex(index=BACKBONES_ORDER, columns=METRIC_ORDER))

    n_metrics   = len(METRIC_ORDER)
    n_backbones = len(BACKBONES_ORDER)

    sector_width = 2 * np.pi / n_metrics
    inter_gap = 0.10 * sector_width
    usable = sector_width - inter_gap
    bar_width = usable / n_backbones

    for j, bb in enumerate(BACKBONES_ORDER):
        if bb not in pivot_mean.index:
            continue
        means = pivot_mean.loc[bb].values
        stds  = pivot_std.loc[bb].values
        m_scaled, s_scaled = [], []
        for k, m, s in zip(METRIC_ORDER, means, stds):
            s = 0.0 if np.isnan(s) else s
            if k == 'silhouette_score':
                m_scaled.append(SILHOUETTE_SCALE(m))
                s_scaled.append(s / 0.5)
            else:
                m_scaled.append(np.clip(m, 0, 1))
                s_scaled.append(s)
        m_scaled = np.array(m_scaled)
        s_scaled = np.array(s_scaled)

        thetas = np.array([
            i * sector_width + inter_gap / 2 + (j + 0.5) * bar_width
            for i in range(n_metrics)
        ])

        ax.bar(thetas, m_scaled, width=bar_width * 0.96,
               color=BACKBONE_COLOURS[bb],
               edgecolor='white', linewidth=0.7,
               alpha=0.65,
               label=bb, zorder=3)

        for t, m, s in zip(thetas, m_scaled, s_scaled):
            lo = max(0.0, m - s)
            hi = min(1.0, m + s)
            ax.plot([t, t], [lo, hi], color='#333333',
                    linewidth=0.9, solid_capstyle='butt', zorder=4)

    # ----- ticks and labels ------------------------------------------------
    label_thetas = np.array([
        i * sector_width + sector_width / 2 for i in range(n_metrics)
    ])
    ax.set_xticks(label_thetas)
    ax.set_xticklabels([])
    for theta, m in zip(label_thetas, METRIC_ORDER):
        r_label = 1.08 if m == 'silhouette_score' else 1.04
        ax.text(theta, r_label, METRIC_LABELS[m],
                ha='center', va='center',
                fontsize=13, weight='bold',
                color='#222222', zorder=5)

    silhouette_idx = METRIC_ORDER.index('silhouette_score')
    boundary_theta = silhouette_idx * sector_width
    ax.set_yticks([0.25, 0.50, 0.75, 1.0])
    ax.set_yticklabels([])
    for r in [0.25, 0.50, 0.75, 1.0]:
        ax.text(boundary_theta, r, f'{r:.2f}'.lstrip('0'),
                ha='center', va='center', fontsize=11,
                color='#666666',
                bbox=dict(facecolor='white', edgecolor='none',
                          boxstyle='round,pad=0.20'),
                zorder=0)

    ax.set_ylim(0, 1.0)
    ax.set_rorigin(-0.35)
    ax.spines['polar'].set_visible(False)
    ax.grid(True, color='#dddddd', linewidth=0.6, zorder=1)
    ax.tick_params(axis='x', which='both', length=0)
    ax.tick_params(axis='y', which='both', length=0)

    ax.set_theta_offset(np.pi / 2 - sector_width / 2)
    ax.set_theta_direction(-1)

    if title:
        ax.set_title(title, fontsize=24, pad=80, weight='bold', color='#222222')


# ----- driver: generate every (setup, eval_dataset, cell) panel -----------
def generate_all_panels(out_dir='figures/radial_bars'):
    os.makedirs(out_dir, exist_ok=True)
    saved = []
    for setup in SETUPS_ORDER:
        eval_datasets = sorted(
            agg[agg.setup == setup].eval_dataset.unique(),
            key=lambda e: list(EVAL_TITLES).index(e) if e in EVAL_TITLES else 999,
        )
        n_rows = len(eval_datasets)
        n_cols = len(CELLS_ORDER)
        if n_rows == 0:
            continue
        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(5.0 * n_cols, 5.0 * n_rows),
            subplot_kw={'projection': 'polar'},
            squeeze=False,
        )
        for i, ev in enumerate(eval_datasets):
            for j, cell in enumerate(CELLS_ORDER):
                ax = axes[i, j]
                radial_bar_panel(ax, setup, ev, cell,
                                 title=CELL_TITLES[cell] if i == 0 else None)
            axes[i, 0].text(
                -0.30, 0.5, EVAL_TITLES.get(ev, ev),
                transform=axes[i, 0].transAxes,
                rotation=90, va='center', ha='center',
                fontsize=24, weight='bold', color='#222222',
            )

        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center',
                   ncol=len(BACKBONES_ORDER),
                   bbox_to_anchor=(0.5, -0.05), frameon=False, fontsize=26)
        fig.suptitle(setup, y=1.00, fontsize=26, weight='bold', color='#222222')

        plt.tight_layout()
        slug = (setup.lower()
                     .replace(' ', '_')
                     .replace('(', '')
                     .replace(')', '')
                     .replace('/', ''))
        fname = os.path.join(out_dir, f'radial_bars_{slug}.pdf')
        plt.savefig(fname)
        saved.append(fname)
        plt.show()
    return saved


saved = generate_all_panels()
print(f'\nSaved {len(saved)} figures:')
for s in saved:
    print(f'  {s}')
